# CA Fire Exposure Processing (FRAP → CASTNET)

**Generated:** 2026-03-08

This notebook builds **site-specific wildfire exposure features** from **CAL FIRE FRAP fire perimeters** and merges them into your project outputs.

It produces two dashboard/model-ready artifacts:

1. `dashboard_data/site_fire_exposure_annual.parquet`  → for vegetation + dashboard
2. `dashboard_data/site_fire_exposure_daily.parquet`   → for model features (merge into hourly data by date)

## Requirements
Install geospatial deps (one-time):

```bash
pip install geopandas shapely fiona pyproj pyarrow pandas
```

## Inputs
- FRAP FileGDB folder (e.g. `fire241.gdb/`)
- `dashboard_data/site_metadata.json` containing site lat/lon

## Notes
- We use **EPSG:3310** (California Albers) for correct buffering and area.
- Daily features use **ALARM_DATE → CONT_DATE** as the active window.


In [1]:
# ----------------------------
# Config
# ----------------------------
from pathlib import Path

# Point this at your FRAP .gdb *folder*
GDB_PATH = Path(r"C:\MATH699P\Data\fire24_1.gdb")  # change if needed
FIRE_LAYER = "firep24_1"        # change if your layer name differs

# Project folders
DASH_DIR = Path(r"C:\MATH699P\Data\dashboard_data")
DASH_DIR.mkdir(exist_ok=True)

# Exposure configuration
BUFFER_KM = 50                  # common values: 25 / 50 / 100
THRESH_KM2 = 5.0                # annual "fire year" if burned_km2 >= this threshold

# Output paths
OUT_ANNUAL = DASH_DIR / "site_fire_exposure_annual.parquet"
OUT_DAILY  = DASH_DIR / "site_fire_exposure_daily.parquet"

# Merge target (optional)
ANNUAL_VEG_PATH = DASH_DIR / "annual_vegetation.parquet"

print("GDB_PATH:", GDB_PATH.resolve())
print("DASH_DIR:", DASH_DIR.resolve())


GDB_PATH: C:\MATH699P\Data\fire24_1.gdb
DASH_DIR: C:\MATH699P\Data\dashboard_data


In [2]:
# ----------------------------
# Load libraries
# ----------------------------
import json
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point

import fiona
print("Fiona version:", fiona.__version__)
print("GeoPandas version:", gpd.__version__)


Fiona version: 1.10.1
GeoPandas version: 1.0.1


In [3]:
# ----------------------------
# Inspect available layers in the .gdb
# ----------------------------
layers = fiona.listlayers(str(GDB_PATH))
layers


['rxburn24_1', 'firep24_1']

If the layer names differ, set `FIRE_LAYER` accordingly. Typically `firep...` is the fire perimeter layer.


In [4]:
# ----------------------------
# Read FRAP fire perimeters
# ----------------------------
fires = gpd.read_file(str(GDB_PATH), layer=FIRE_LAYER)
fires.shape, fires.columns.tolist()[:30]


c:\Users\snehi\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


((22810, 20),
 ['YEAR_',
  'STATE',
  'AGENCY',
  'UNIT_ID',
  'FIRE_NAME',
  'INC_NUM',
  'IRWINID',
  'ALARM_DATE',
  'CONT_DATE',
  'C_METHOD',
  'CAUSE',
  'COMPLEX_NAME',
  'COMPLEX_ID',
  'OBJECTIVE',
  'GIS_ACRES',
  'COMMENTS',
  'FIRE_NUM',
  'Shape_Length',
  'Shape_Area',
  'geometry'])

## Identify date columns
We expect **ALARM_DATE** and **CONT_DATE** (containment). If your field names differ, update them below.


In [5]:
# ----------------------------
# Normalize date fields
# ----------------------------
# Adjust these if your layer uses different names
ALARM_COL = "ALARM_DATE"
CONT_COL  = "CONT_DATE"

if ALARM_COL not in fires.columns or CONT_COL not in fires.columns:
    raise KeyError(f"Expected columns not found. Available columns include: {fires.columns.tolist()}")

fires["alarm_date"] = pd.to_datetime(fires[ALARM_COL], errors="coerce")
fires["cont_date"]  = pd.to_datetime(fires[CONT_COL], errors="coerce")

# Some fires may have missing containment; cap them to alarm + 120d to keep windows finite
mask = fires["cont_date"].isna() & fires["alarm_date"].notna()
fires.loc[mask, "cont_date"] = fires.loc[mask, "alarm_date"] + pd.Timedelta(days=120)

fires = fires.dropna(subset=["alarm_date", "cont_date"]).copy()
fires["year"] = fires["alarm_date"].dt.year.astype(int)

# Basic sanity checks
fires[["alarm_date","cont_date","year"]].describe(include="all")


,alarm_date,cont_date,year
count,17414,17414,17414.000000
mean,1990-11-15 15:55:55.231000+00:00,1991-01-09 18:22:56.823000+00:00,1990.298266
min,1898-04-01 00:00:00+00:00,1898-07-30 00:00:00+00:00,1898.000000
25%,1972-10-24 00:00:00+00:00,1973-02-21 00:00:00+00:00,1972.000000
50%,1998-06-10 12:00:00+00:00,1998-06-21 00:00:00+00:00,1998.000000
75%,2013-08-19 00:00:00+00:00,2013-09-06 18:00:00+00:00,2013.000000
max,2025-01-22 00:00:00+00:00,2025-02-04 00:00:00+00:00,2025.000000
std,NaN,NaN,27.341416


## Load CASTNET site metadata
Must contain `lat` and `lon` per site.


In [6]:
site_meta_path = DASH_DIR / "site_metadata.json"
if not site_meta_path.exists():
    raise FileNotFoundError(f"Missing {site_meta_path}. Put your site_metadata.json in dashboard_data/")

with open(site_meta_path, "r") as f:
    site_meta = json.load(f)

# Build GeoDataFrame of sites
site_rows = []
missing = []
for sid, v in site_meta.items():
    if not isinstance(v, dict) or "lat" not in v or "lon" not in v:
        missing.append(sid)
        continue
    site_rows.append({"SITE_ID": sid, "lat": v["lat"], "lon": v["lon"], "geometry": Point(v["lon"], v["lat"])})

if missing:
    print("Warning: these sites were missing lat/lon and were skipped:", missing)

sites = gpd.GeoDataFrame(site_rows, crs="EPSG:4326")
sites.head(), sites.shape


(  SITE_ID      lat       lon                   geometry
 0  SEK402  36.4886 -118.8284  POINT (-118.8284 36.4886)
 1  YOS404  37.7133 -119.7069  POINT (-119.7069 37.7133)
 2  JOT403  34.0617 -116.3873  POINT (-116.3873 34.0617)
 3  LAV410  40.5402 -121.5763  POINT (-121.5763 40.5402)
 4  TRI193  40.9603 -122.9547  POINT (-122.9547 40.9603),
 (10, 4))

## Reproject + buffer
We use EPSG:3310 to compute buffers and areas in meters.


In [7]:
# Project to CA Albers (meters)
fires_3310 = fires.to_crs(epsg=3310)
sites_3310 = sites.to_crs(epsg=3310)

# Buffer each site
sites_buf = sites_3310.copy()
sites_buf["geometry"] = sites_buf.geometry.buffer(BUFFER_KM * 1000)

fires_3310.crs, sites_buf.crs


(<Projected CRS: EPSG:3310>
 Name: NAD83 / California Albers
 Axis Info [cartesian]:
 - X[east]: Easting (metre)
 - Y[north]: Northing (metre)
 Area of Use:
 - name: United States (USA) - California.
 - bounds: (-124.45, 32.53, -114.12, 42.01)
 Coordinate Operation:
 - name: California Albers
 - method: Albers Equal Area
 Datum: North American Datum 1983
 - Ellipsoid: GRS 1980
 - Prime Meridian: Greenwich,
 <Projected CRS: EPSG:3310>
 Name: NAD83 / California Albers
 Axis Info [cartesian]:
 - X[east]: Easting (metre)
 - Y[north]: Northing (metre)
 Area of Use:
 - name: United States (USA) - California.
 - bounds: (-124.45, 32.53, -114.12, 42.01)
 Coordinate Operation:
 - name: California Albers
 - method: Albers Equal Area
 Datum: North American Datum 1983
 - Ellipsoid: GRS 1980
 - Prime Meridian: Greenwich)

## Spatial intersection
We intersect fire polygons with each site buffer to estimate burned area *within the buffer*.


In [8]:
# Intersection
ix = gpd.overlay(
    fires_3310[["alarm_date", "cont_date", "year", "geometry"]],
    sites_buf[["SITE_ID", "geometry"]],
    how="intersection",
)

ix["burned_m2_in_buffer"] = ix.geometry.area
ix["burned_km2_in_buffer"] = ix["burned_m2_in_buffer"] / 1e6

ix[["SITE_ID","year","alarm_date","cont_date","burned_km2_in_buffer"]].head(), ix.shape


(  SITE_ID  year                alarm_date                 cont_date  \
 0  SAN405  2025 2025-01-08 00:00:00+00:00 2025-01-31 00:00:00+00:00   
 1  SAN405  2025 2025-01-08 00:00:00+00:00 2025-01-11 00:00:00+00:00   
 2  LAV410  2024 2024-07-24 00:00:00+00:00 2024-09-07 00:00:00+00:00   
 3  SAN405  2024 2024-09-08 00:00:00+00:00 2024-12-31 00:00:00+00:00   
 4  SND152  2024 2024-09-06 00:00:00+00:00 2024-12-23 00:00:00+00:00   
 
    burned_km2_in_buffer  
 0             56.883667  
 1              0.740738  
 2            811.236136  
 3            225.507725  
 4            177.962977  ,
 (4958, 7))

## Annual output (vegetation + dashboard)


In [9]:
annual = (
    ix.groupby(["SITE_ID", "year"], as_index=False)
      .agg(
          burned_km2_50km=("burned_km2_in_buffer", "sum"),
          n_fires_50km=("burned_km2_in_buffer", "size"),
      )
)

annual["is_fire_year_50km"] = annual["burned_km2_50km"] >= THRESH_KM2

annual.to_parquet(OUT_ANNUAL, index=False)
print("Wrote:", OUT_ANNUAL, "| rows:", len(annual))
annual.sort_values(["SITE_ID","year"]).head(10)


Wrote: C:\MATH699P\Data\dashboard_data\site_fire_exposure_annual.parquet | rows: 776


,SITE_ID,year,burned_km2_50km,n_fires_50km,is_fire_year_50km
0,ABF404,1950,62.147258,5,True
1,ABF404,1951,91.365215,5,True
2,ABF404,1952,22.163008,2,True
3,ABF404,1953,41.144451,6,True
4,ABF404,1954,39.275803,3,True
5,ABF404,1955,18.300355,5,True
6,ABF404,1956,105.228308,5,True
7,ABF404,1957,26.765868,6,True
8,ABF404,1958,39.506759,13,True
9,ABF404,1959,67.648449,7,True


## Daily output (model features)
We create a **daily time series per site** representing the sum of burned area from fires **active on that day** (alarm→containment). Then we compute rolling sums (7/30/90 days).


In [10]:
# Build daily exposure using an efficient "delta sweep" (+area on start, -area after end)
ix_small = ix[["SITE_ID", "alarm_date", "cont_date", "burned_km2_in_buffer"]].copy()
ix_small["alarm_date"] = pd.to_datetime(ix_small["alarm_date"]).dt.floor("D")
ix_small["cont_date"]  = pd.to_datetime(ix_small["cont_date"]).dt.floor("D")

daily_frames = []
count_frames = []

for sid, g in ix_small.groupby("SITE_ID"):
    # area deltas
    area_deltas = {}
    cnt_deltas = {}
    for r in g.itertuples(index=False):
        start = r.alarm_date
        end = r.cont_date
        area = float(r.burned_km2_in_buffer)

        area_deltas[start] = area_deltas.get(start, 0.0) + area
        area_deltas[end + pd.Timedelta(days=1)] = area_deltas.get(end + pd.Timedelta(days=1), 0.0) - area

        cnt_deltas[start] = cnt_deltas.get(start, 0) + 1
        cnt_deltas[end + pd.Timedelta(days=1)] = cnt_deltas.get(end + pd.Timedelta(days=1), 0) - 1

    if not area_deltas:
        continue

    all_dates = pd.date_range(min(area_deltas.keys()), max(area_deltas.keys()), freq="D")

    s_area = pd.Series(0.0, index=all_dates)
    for d, v in area_deltas.items():
        if d in s_area.index:
            s_area.loc[d] += v
    active_area = s_area.cumsum()

    s_cnt = pd.Series(0, index=all_dates)
    for d, v in cnt_deltas.items():
        if d in s_cnt.index:
            s_cnt.loc[d] += v
    active_cnt = s_cnt.cumsum()

    out = pd.DataFrame({
        "date": active_area.index,
        "SITE_ID": sid,
        "fire_km2_active_today_50km": active_area.values,
        "n_active_fires_today_50km": active_cnt.values.astype(int),
    })
    daily_frames.append(out)

fire_daily = pd.concat(daily_frames, ignore_index=True) if daily_frames else pd.DataFrame(
    columns=["date","SITE_ID","fire_km2_active_today_50km","n_active_fires_today_50km"]
)

fire_daily = fire_daily.sort_values(["SITE_ID","date"])

# Rolling sums (great ML features)
for w in [7, 30, 90]:
    fire_daily[f"fire_km2_active_{w}d_50km"] = (
        fire_daily.groupby("SITE_ID")["fire_km2_active_today_50km"]
        .rolling(w, min_periods=1).sum()
        .reset_index(level=0, drop=True)
    )

fire_daily.to_parquet(OUT_DAILY, index=False)
print("Wrote:", OUT_DAILY, "| rows:", len(fire_daily))
fire_daily.head()


Wrote: C:\MATH699P\Data\dashboard_data\site_fire_exposure_daily.parquet | rows: 329757


,date,SITE_ID,fire_km2_active_today_50km,n_active_fires_today_50km,fire_km2_active_7d_50km,fire_km2_active_30d_50km,fire_km2_active_90d_50km
0,1950-06-01 00:00:00+00:00,ABF404,1.88726,1,1.88726,1.88726,1.88726
1,1950-06-02 00:00:00+00:00,ABF404,1.88726,1,3.77452,3.77452,3.77452
2,1950-06-03 00:00:00+00:00,ABF404,1.88726,1,5.66178,5.66178,5.66178
3,1950-06-04 00:00:00+00:00,ABF404,1.88726,1,7.54904,7.54904,7.54904
4,1950-06-05 00:00:00+00:00,ABF404,1.88726,1,9.43630,9.43630,9.43630


## Merge into annual vegetation (optional)
If you already export `annual_vegetation.parquet`, this merges annual fire exposure into it.


In [11]:
if ANNUAL_VEG_PATH.exists():
    veg = pd.read_parquet(ANNUAL_VEG_PATH)
    fire_annual = pd.read_parquet(OUT_ANNUAL)

    # Drop fire columns if already present from a previous run — prevents
    # pandas creating _x/_y suffixed duplicates on re-merge
    fire_cols = ["burned_km2_50km", "n_fires_50km", "is_fire_year_50km"]
    veg = veg.drop(columns=[c for c in fire_cols if c in veg.columns])

    merged = veg.merge(
        fire_annual,
        on=["SITE_ID","year"],
        how="left"
    )

    merged["burned_km2_50km"] = merged["burned_km2_50km"].fillna(0.0)
    merged["n_fires_50km"] = merged["n_fires_50km"].fillna(0).astype(int)
    merged["is_fire_year_50km"] = merged["is_fire_year_50km"].fillna(False)

    merged.to_parquet(ANNUAL_VEG_PATH, index=False)
    print("Updated annual vegetation file:", ANNUAL_VEG_PATH)
    merged.head()
else:
    print("Skipping merge: annual_vegetation.parquet not found at", ANNUAL_VEG_PATH)


Updated annual vegetation file: C:\MATH699P\Data\dashboard_data\annual_vegetation.parquet


C:\Users\snehi\AppData\Local\Temp\ipykernel_12224\3149244182.py:18: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged["is_fire_year_50km"] = merged["is_fire_year_50km"].fillna(False)


## How to use daily fire features in the forecasting model
In your **feature engineering** notebook, merge by `SITE_ID` and **date**:

```python
df_hourly['date'] = pd.to_datetime(df_hourly['DATE_TIME']).dt.floor('D')
fire_daily = pd.read_parquet('dashboard_data/site_fire_exposure_daily.parquet')
df_hourly = df_hourly.merge(fire_daily, on=['SITE_ID','date'], how='left')
fire_cols = [c for c in df_hourly.columns if c.startswith('fire_') or c.startswith('n_active_fires')]
df_hourly[fire_cols] = df_hourly[fire_cols].fillna(0)
```
